# 03 · Hardware: inference on ibm_kingston

**Settings:** Accelerator *None*, Internet **on**, Add-ons → **Secrets**: `IBM_QUANTUM_TOKEN` (your API key) and,
optionally, `IBM_QUANTUM_INSTANCE` (the CRN of your Open Plan instance).
**Inputs:** `bbd-code`, the output of notebook 01 and the output of notebook 02 (it needs the checkpoints of the
two hardware models, which the grid trains first).

Run it once with `DRY_RUN = True`: it does everything against a simulated `ibm_kingston` (FakeKingston) with no
token and no cost. Then set `DRY_RUN = False`. The real run submits a small **cost probe** first, measures the
QPU seconds per shot from it, and sizes the main job to fit `MINUTES_AVAILABLE` of your Open Plan allowance
(10 minutes per 28 days). The main job carries every circuit (LA model in layouts A and B, the CE model with
post-hoc adjustment in layout A, and two readout-calibration circuits) so all conditions share one calibration
window. Job ids are saved to `/kaggle/working/hardware/jobs.json`; if the queue outlasts the session, set
`RESUME_JOB` to the id and run again.

In [ ]:
# Pinned: these are the versions the hardware code was tested with (SamplerV2 API).
%pip install -q "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0" "qiskit-aer==0.18.0" pylatexenc

In [ ]:
# --- bootstrap: find the bbd-code dataset, copy it to a writable folder, import it
import glob, os, shutil, sys
# only the bbd-code dataset counts: attached notebook outputs never contain the code (it is copied to /tmp)
hits = [h for h in sorted(glob.glob("/kaggle/input/**/bbd/__init__.py", recursive=True)) if "/bbd_code/" not in h]
hits = hits or sorted(glob.glob("./bbd/__init__.py"))
if not hits:
    raise FileNotFoundError("Attach the 'bbd-code' dataset to this notebook (KAGGLE_INSTRUCTIONS.md, step 2).")
if len(hits) > 1:
    print("warning: several copies of the code are attached; using", hits[0])
SRC = os.path.dirname(os.path.dirname(os.path.abspath(hits[0])))
CODE = "/tmp/bbd_code" if os.path.isdir("/kaggle/working") else SRC
if CODE != SRC:
    shutil.rmtree(CODE, ignore_errors=True)
    shutil.copytree(SRC, CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", "results", "work"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
import bbd
print("bbd", bbd.__version__, "loaded from", SRC)

In [ ]:
DRY_RUN = True                 # True: FakeKingston, no token, no cost.  False: the real ibm_kingston
BACKEND = "ibm_kingston"
N_TEST, N_CAL = 600, 50        # stratified test inputs, validation inputs for the affine calibration
MINUTES_AVAILABLE = 8.0        # QPU minutes you are willing to spend on the main job
RESUME_JOB = None              # paste a main-job id to fetch a job submitted in an earlier session (real runs only;
                               # attach that session's output so its hardware/jobs.json is found)
MODELS = {"LA": "toniot8_r100_Q_s3_LA_seed0", "CE": "toniot8_r100_Q_s3_CE_seed0"}
POSTHOC = ("CE",)              # the CE model is evaluated with post-hoc adjustment (H5)
if DRY_RUN:                    # the noisy simulator is slow: a smaller run checks the whole pipeline
    N_TEST, N_CAL = 90, 27
    RESUME_JOB = None

In [ ]:
import json, time, numpy as np
from bbd import config, hardware as hw, runner, store
RESULTS = config.RESULTS_DIR
runner.carry_forward(RESULTS, config.INPUT_ROOT, kinds=("ledger",))   # artifacts are read from the inputs in place
SAVED = None
if RESUME_JOB:                 # analyse the job exactly as it was submitted
    SAVED = hw.find_job(RESUME_JOB)
    BACKEND, N_TEST, N_CAL, MODELS = SAVED["backend"], SAVED["n_test"], SAVED["n_cal"], SAVED["models"]
    print("resuming", RESUME_JOB, "submitted", SAVED["submitted"], "on", BACKEND, "path", SAVED["path"])
missing = [r for r in MODELS.values() if not store.has(RESULTS, "ckpt", r)]
assert not missing, f"checkpoints missing: {missing}. Attach the output of notebook 02 (its first version trains these)."
if DRY_RUN:
    service, backend = None, hw.get_backend(name=BACKEND, fake=True)
else:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    token = sec.get_secret("IBM_QUANTUM_TOKEN")
    try:
        instance = sec.get_secret("IBM_QUANTUM_INSTANCE")
    except Exception:
        instance = None
    service = hw.connect(token, instance)
    backend = service.backend(BACKEND)
print(backend.name, json.dumps(hw.describe_account(service, backend) if service else {"mode": "dry run"}, default=str, indent=1))

In [ ]:
inputs = {k: hw.model_inputs(RESULTS, rid, N_TEST, N_CAL) for k, rid in MODELS.items()}
for k, inp in inputs.items():
    print(k, inp["run_id"], "| s =", inp["s"], "| test", len(inp["y_test"]), "| cal", len(inp["y_cal"]),
          "| max |sim - Qiskit| = %.1e" % hw.verify_against_qiskit(inp))

In [ ]:
bp = hw.best_path(backend) if SAVED is None else hw.path_info(backend, SAVED["path"])   # resumed: the submitted path
path = bp["path"]
print("path:", path); print("readout errors:", np.round(bp["readout"], 4)); print("CZ errors:", np.round(bp["cz"], 4))
circ, isa = {}, {}
for k, inp in inputs.items():
    circ[k] = hw.build_circuit(inp["phi"], inp["k"], inp["entangle"], inp["reupload"])
    for lay, layout in (("A", path), ("B", path[::-1])):
        isa[(k, lay)] = hw.to_isa(circ[k], backend, layout)
        chk = hw.check_isa(isa[(k, lay)], circ[k], layout)
        print(k, lay, {x: chk[x] for x in ("ok", "two_qubit_isa", "depth")})
        assert chk["ok"], chk
c0, c1 = hw.calibration_circuits(len(path))
cal_isa = [hw.to_isa(c0, backend, path), hw.to_isa(c1, backend, path)]
config.HW_DIR.mkdir(parents=True, exist_ok=True)
(config.HW_DIR / "path.json").write_text(json.dumps(dict(bp, backend=backend.name, time=time.ctime()), default=str, indent=1))

## Cost probe (20 inputs × 1,024 shots)

In [ ]:
if RESUME_JOB is None:
    th = inputs["LA"]["theta_test"][:20]
    probe = hw.submit(backend, [(isa[("LA", "A")], th, 256 if DRY_RUN else 1024)], "probe", dict(n=20, shots=1024))
    print("probe job:", probe.job_id()); pr = probe.result()
    used = hw.job_usage_seconds(probe)
    per_shot = (used / (20 * 1024)) if used else 3e-4
    print(f"usage {used} s  ->  {per_shot * 1e6:.0f} us per shot (fallback 300 us if unreported)")
    n_inputs = len(inputs["LA"]["theta_test"]) + len(inputs["LA"]["theta_cal"])
    SHOTS = 256 if DRY_RUN else hw.plan_shots(per_shot, n_inputs, n_pubs=3, minutes_available=MINUTES_AVAILABLE)
    est_min = per_shot * SHOTS * n_inputs * 3 / 60
    print(f"main job: {SHOTS} shots per input, about {est_min:.1f} QPU minutes")

## Main job

In [ ]:
if RESUME_JOB is None:
    pubs, order = [], []
    for k, lay in (("LA", "A"), ("LA", "B"), ("CE", "A")):
        inp = inputs[k]
        pubs.append((isa[(k, lay)], np.vstack([inp["theta_test"], inp["theta_cal"]]), SHOTS)); order.append((k, lay))
    pubs += [(cal_isa[0], None, 2048 if DRY_RUN else 8192), (cal_isa[1], None, 2048 if DRY_RUN else 8192)]
    job = hw.submit(backend, pubs, "main", dict(order=order, shots=SHOTS, path=path, backend=backend.name,
                                                n_test=N_TEST, n_cal=N_CAL, models=MODELS, dry_run=DRY_RUN))
    RESUME_JOB = job.job_id()
    print("MAIN JOB ID:", RESUME_JOB, " (saved in hardware/jobs.json)")
else:
    job = service.job(RESUME_JOB)
    order, SHOTS = [tuple(o) for o in SAVED["order"]], SAVED["shots"]
t0 = time.time()
while not DRY_RUN and str(job.status()) not in ("DONE", "JobStatus.DONE", "ERROR", "CANCELLED"):
    print(time.strftime("%H:%M"), job.status()); time.sleep(120)
res = job.result()
print("done after %.0f min; usage %s s" % ((time.time() - t0) / 60, hw.job_usage_seconds(job)))

In [ ]:
hw_z = {}
for i, (k, lay) in enumerate(order):
    nt = len(inputs[k]["theta_test"])
    z = hw.counts_to_z(res[i].data.c, inputs[k]["k"])
    hw_z[(k, lay)] = (z[:nt], z[nt:])
cal = {"z0": hw.counts_to_z(res[len(order)].data.c, len(path))[0], "z1": hw.counts_to_z(res[len(order) + 1].data.c, len(path))[0]}
out = hw.analyse(inputs, hw_z, path, cal, SHOTS, posthoc_keys=POSTHOC)
import pandas as pd
tab = pd.DataFrame({name: {c: round(v["macro_f1"], 3) for c, v in conds.items()} for name, conds in out["conditions"].items()}).T
display(tab)
print("H4  Spearman(beta_k, over-prediction):", out.get("spearman"))
print("H4  over-prediction follows the qubit across layouts:", out.get("follows_qubit"))
print("H5  tail predictions, post-hoc model:", json.dumps(hw.h5_tail_inflation(out), indent=1))

In [ ]:
from bbd import figures
from IPython.display import Image, display
for f in (figures.fig_hw_conditions, figures.fig_hw_bias, figures.fig_hw_confusion):
    for p in f(out, config.FIG_DIR):
        if str(p).endswith(".png"): display(Image(str(p), width=900))
hw.draw_circuit((1, 10, 2), inputs["LA"]["k"], config.FIG_DIR)